# 33 — Incremental logistic analysis

Tests whether genomics adds predictive information beyond ZeBRA using the **exact cohort definition from notebook 32**: left-merge adjudicated targets; numeric conversion with missing targets set to 0; left-merge `predicted_risk`; retain non-missing `predicted_risk`; 30 repeated stratified 40% train / 60% held-out splits.

Primary comparison: ZeBRA-only logistic vs nested elastic-net logistic using ZeBRA + all genomic features. Secondary exploratory models add MUC5B (`rs35705950`) and MUC5B+DOCK2 (`rs13183751`). Hyperparameters are selected only within each outer-training split. Outputs include paired ΔAUC/AP/Brier/log-loss and coefficient stability.

Low-FPR operating points at 0.5%, 1%, and 5% FPR are also compared pairwise.

In [7]:
import json,secrets,warnings
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.model_selection import train_test_split,StratifiedKFold,GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score,roc_curve,average_precision_score,brier_score_loss,log_loss
warnings.filterwarnings("ignore")

N_REPEATS=30; TRAIN_SIZE=.40; INNER_CV_FOLDS=3; OPERATING_FPRS=[.005,.01,.05]
BASE_DATA_FILE="ILD_TOP_DRIVERS_DATA.csv"
TARGET_FILE="REPHENOTYPES FOR IC.csv"
PRED_FILE="PREDICTIONS_104W_PRED_WINDOW.parquet"
OUT=Path("./RESULTS/033_INCREMENTAL_LOGISTIC_32_COHORT"); OUT.mkdir(parents=True,exist_ok=True)

BASE_DATA=pd.read_csv(BASE_DATA_FILE).drop(columns=["target"],errors="ignore")
NEW_TARGETS=(pd.read_csv(TARGET_FILE)
 .replace({"N":0,"Y":1,"y":1,"n":0,"na":np.nan})
 .rename(columns={"arb_person_id":"patient_id"}))
PREDS=pd.read_parquet(PRED_FILE)
TARGET_NAMES=[c for c in NEW_TARGETS.columns if c!="patient_id"]

def cohort(target_name):
    d=(BASE_DATA
       .merge(NEW_TARGETS[["patient_id",target_name]].rename(columns={target_name:"target"}),
              on="patient_id",how="left")
       .merge(PREDS[["patient_id","predicted_risk"]],on="patient_id",how="left"))
    d["target"]=pd.to_numeric(d["target"],errors="coerce").fillna(0).astype(int)
    return d[d["predicted_risk"].notnull()].copy()

# exact notebook-32 sanity check
san=[]
for t in TARGET_NAMES:
    d=cohort(t); X=d.drop(columns=["patient_id","target"]); y=d["target"]
    _,Xv,_,yv=train_test_split(X,y,train_size=.40,stratify=y,random_state=1321)
    san.append({"target_name":t,"n_total":len(d),"n_positive_total":int(y.sum()),
                "n_validation":len(yv),"n_positive_validation":int(yv.sum()),
                "zebra_auc_exact_32_sanity_split":roc_auc_score(yv,Xv["predicted_risk"])})
SANITY=pd.DataFrame(san); SANITY.to_csv(OUT/"sanity_exact_32_cohort.csv",index=False)
display(SANITY)

,target_name,n_total,n_positive_total,n_validation,n_positive_validation,zebra_auc_exact_32_sanity_split
0,FILD or FILA ADJUDICATED,12825,254,7695,152,0.804147
1,NONFIBROTIC ILD ADJUDICATED,12825,27,7695,16,0.837902
2,NON FIBROTIC ILA ADJUDICATED,12825,74,7695,44,0.809235


In [8]:
def num(X): return X.apply(pd.to_numeric,errors="coerce")

def baseline(seed):
    return Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler()),
      ("lr",LogisticRegression(C=1e6,solver="lbfgs",max_iter=5000,random_state=seed))])

def nested(X,y,seed,kind="elastic"):
    X=num(X).reset_index(drop=True); y=np.asarray(y,int)
    k=min(INNER_CV_FOLDS,int(np.bincount(y,minlength=2).min()))
    if k<2: raise ValueError("Too few minority-class observations for inner CV")
    cv=StratifiedKFold(k,shuffle=True,random_state=seed)
    if kind=="elastic":
        pipe=Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("sc",StandardScaler()),
          ("lr",LogisticRegression(penalty="elasticnet",solver="saga",max_iter=20000,
                                   tol=1e-4,random_state=seed))])
        grid={"lr__C":[.001,.01,.1],"lr__l1_ratio":[0,.5,1]}
    else:
        pipe=Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("sc",StandardScaler()),
          ("lr",LogisticRegression(penalty="l2",solver="liblinear",max_iter=10000,
                                   random_state=seed))])
        grid={"lr__C":[.01,.1,1,10]}
    g=GridSearchCV(pipe,grid,scoring="roc_auc",cv=cv,n_jobs=-1,refit=True)
    g.fit(X,y); return g.best_estimator_,g.best_params_,float(g.best_score_)

def metrics(y,p):
    p=np.asarray(p,float); q=np.clip(p,1e-8,1-1e-8)
    return {"auc":roc_auc_score(y,p),"average_precision":average_precision_score(y,p),
            "brier":brier_score_loss(y,q),"log_loss":log_loss(y,q,labels=[0,1])}


def opmetric(y,p,requested_fpr):
    fpr,tpr,thr=roc_curve(y,p,drop_intermediate=False)
    i=int(np.argmin(np.abs(fpr-requested_fpr))); cutoff=thr[i]
    pred=np.asarray(p)>=cutoff; y=np.asarray(y,int)
    tp=((pred==1)&(y==1)).sum(); fp=((pred==1)&(y==0)).sum()
    tn=((pred==0)&(y==0)).sum(); fn=((pred==0)&(y==1)).sum()
    sens=tp/(tp+fn) if tp+fn else np.nan; spec=tn/(tn+fp) if tn+fp else np.nan
    ppv=tp/(tp+fp) if tp+fp else np.nan
    return {"requested_fpr":requested_fpr,"actual_fpr":1-spec,
            "sensitivity":sens,"specificity":spec,"ppv":ppv}

def coefs(model,cols):
    try: names=list(model.named_steps["imp"].get_feature_names_out(cols))
    except: names=list(cols)
    b=np.asarray(model.named_steps["lr"].coef_).ravel()
    if len(names)!=len(b): names=[f"feature_{i}" for i in range(len(b))]
    return pd.DataFrame({"feature":names,"coef":b,"abs_coef":np.abs(b),
                         "nonzero":np.abs(b)>1e-10})

In [9]:
from tqdm import tqdm

In [10]:
rows=[]; ops=[]; hp=[]; cf=[]
for t in TARGET_NAMES:
    d=cohort(t); X=num(d.drop(columns=["patient_id","target"])); y=d["target"].to_numpy(int)
    gcols=[c for c in X.columns if c!="predicted_risk"]
    muc=[c for c in gcols if str(c).startswith("rs35705950")]
    dock=[c for c in gcols if str(c).startswith("rs13183751")]
    print(f"{t}: N={len(d):,}, positives={y.sum():,}, genomic={len(gcols):,}; MUC5B={muc}; DOCK2={dock}")
    for r in tqdm(range(1,N_REPEATS+1)):
        ss=secrets.randbits(31); tr,te=train_test_split(np.arange(len(d)),train_size=TRAIN_SIZE,
                                                       stratify=y,random_state=ss)
        yt,ye=y[tr],y[te]; Xt,Xe=X.iloc[tr],X.iloc[te]
        p_raw=Xe["predicted_risk"].to_numpy(float)
        m0=baseline(secrets.randbits(31)); m0.fit(Xt[["predicted_risk"]],yt)
        p0=m0.predict_proba(Xe[["predicted_risk"]])[:,1]
        m1,bp,cv=nested(Xt,yt,secrets.randbits(31),"elastic")
        p1=m1.predict_proba(Xe)[:,1]
        scores={"zebra_raw":p_raw,"zebra_logistic":p0,"zebra_plus_genomics_elasticnet":p1}
        targeted={}
        if muc: targeted["zebra_plus_MUC5B_ridge"]=["predicted_risk",*muc]
        if muc and dock: targeted["zebra_plus_MUC5B_DOCK2_ridge"]=["predicted_risk",*muc,*dock]
        for name,cols in targeted.items():
            mm,bb,cc=nested(Xt[cols],yt,secrets.randbits(31),"ridge")
            scores[name]=mm.predict_proba(Xe[cols])[:,1]
            hp.append({"target_name":t,"repeat":r,"split_seed":ss,"model":name,
                       "inner_cv_auc":cc,"params_json":json.dumps(bb,sort_keys=True)})
        for name,p in scores.items():
            z={"target_name":t,"repeat":r,"split_seed":ss,"model":name,
               "n_test":len(te),"n_positive_test":int(ye.sum())}; z.update(metrics(ye,p)); rows.append(z)
            for f in OPERATING_FPRS:
                o={"target_name":t,"repeat":r,"split_seed":ss,"model":name}; o.update(opmetric(ye,p,f)); ops.append(o)
        cc=coefs(m1,Xt.columns); cc["target_name"]=t; cc["repeat"]=r; cc["split_seed"]=ss; cf.append(cc)
        hp.append({"target_name":t,"repeat":r,"split_seed":ss,"model":"zebra_plus_genomics_elasticnet",
                   "inner_cv_auc":cv,"params_json":json.dumps(bp,sort_keys=True)})
        #print(" ",r,"/",N_REPEATS)

FILD or FILA ADJUDICATED: N=12,825, positives=254, genomic=513; MUC5B=['rs35705950.1_G_0', 'rs35705950.1_G_1', 'rs35705950.1_G_2']; DOCK2=['rs13183751_G_0', 'rs13183751_G_1', 'rs13183751_G_2']


100%|██████████████████████████████████████████| 30/30 [22:03<00:00, 44.12s/it]


NONFIBROTIC ILD ADJUDICATED: N=12,825, positives=27, genomic=513; MUC5B=['rs35705950.1_G_0', 'rs35705950.1_G_1', 'rs35705950.1_G_2']; DOCK2=['rs13183751_G_0', 'rs13183751_G_1', 'rs13183751_G_2']


  0%|                                                   | 0/30 [01:59<?, ?it/s]


KeyboardInterrupt: 

In [11]:
M=pd.DataFrame(rows); O=pd.DataFrame(ops); C=pd.concat(cf,ignore_index=True); H=pd.DataFrame(hp)
M.to_csv(OUT/"metrics_by_split.csv",index=False); O.to_csv(OUT/"operating_points_by_split.csv",index=False)
C.to_csv(OUT/"coefficients_by_split.csv",index=False); H.to_csv(OUT/"hyperparameters_by_split.csv",index=False)

A=(M.groupby(["target_name","model"])["auc"]
   .agg(n_repeats="size",mean="mean",sd="std",median="median",min="min",max="max").reset_index())
A.to_csv(OUT/"auc_distribution_summary.csv",index=False); display(A)

out=[]
for t,s in M.groupby("target_name"):
    w=s.pivot_table(index=["repeat","split_seed"],columns="model",
                    values=["auc","average_precision","brier","log_loss"],aggfunc="first")
    for met in ["auc","average_precision","brier","log_loss"]:
        d=(w[(met,"zebra_plus_genomics_elasticnet")]-w[(met,"zebra_logistic")]).dropna().to_numpy()
        out.append({"target_name":t,"metric":met,"comparison":"genomics_plus_zebra - zebra",
                    "n_repeats":len(d),"mean_delta":d.mean(),"median_delta":np.median(d),
                    "q025":np.quantile(d,.025),"q975":np.quantile(d,.975),
                    "fraction_gt_zero":np.mean(d>0),"fraction_lt_zero":np.mean(d<0)})
D=pd.DataFrame(out); D.to_csv(OUT/"paired_incremental_summary.csv",index=False); display(D)

,target_name,model,n_repeats,mean,sd,median,min,max
0,FILD or FILA ADJUDICATED,zebra_logistic,30,0.813541,0.012125,0.812887,0.790551,0.836371
1,FILD or FILA ADJUDICATED,zebra_plus_MUC5B_DOCK2_ridge,30,0.811898,0.011454,0.811142,0.791157,0.839064
2,FILD or FILA ADJUDICATED,zebra_plus_MUC5B_ridge,30,0.815697,0.011865,0.815269,0.791355,0.840265
3,FILD or FILA ADJUDICATED,zebra_plus_genomics_elasticnet,30,0.814255,0.011840,0.813194,0.792571,0.836371
4,FILD or FILA ADJUDICATED,zebra_raw,30,0.813541,0.012125,0.812887,0.790551,0.836371


,target_name,metric,comparison,n_repeats,mean_delta,median_delta,q025,q975,fraction_gt_zero,fraction_lt_zero
0,FILD or FILA ADJUDICATED,auc,genomics_plus_zebra - zebra,30,0.000714,0.000000,-0.000145,0.003302,0.366667,0.066667
1,FILD or FILA ADJUDICATED,average_precision,genomics_plus_zebra - zebra,30,-0.021745,0.000000,-0.084443,0.000000,0.000000,0.433333
2,FILD or FILA ADJUDICATED,brier,genomics_plus_zebra - zebra,30,0.000321,0.000324,0.000231,0.000381,1.000000,0.000000
3,FILD or FILA ADJUDICATED,log_loss,genomics_plus_zebra - zebra,30,0.004836,0.004716,0.001832,0.007021,1.000000,0.000000


In [12]:
opout=[]
for (t,f),s in O.groupby(["target_name","requested_fpr"]):
    w=s.pivot_table(index=["repeat","split_seed"],columns="model",
                    values=["sensitivity","ppv"],aggfunc="first")
    for met in ["sensitivity","ppv"]:
        d=(w[(met,"zebra_plus_genomics_elasticnet")]-w[(met,"zebra_logistic")]).dropna().to_numpy()
        opout.append({"target_name":t,"requested_fpr":f,"metric":met,
                      "comparison":"genomics_plus_zebra - zebra","n_repeats":len(d),
                      "mean_delta":d.mean(),"median_delta":np.median(d),
                      "q025":np.quantile(d,.025),"q975":np.quantile(d,.975),
                      "fraction_gt_zero":np.mean(d>0)})
OPD=pd.DataFrame(opout); OPD.to_csv(OUT/"paired_operating_point_summary.csv",index=False); display(OPD)

S=(C.groupby(["target_name","feature"])
   .agg(n_splits=("repeat","nunique"),nonzero_fraction=("nonzero","mean"),
        mean_coef=("coef","mean"),median_coef=("coef","median"),
        mean_abs_coef=("abs_coef","mean"),max_abs_coef=("abs_coef","max"),
        positive_fraction=("coef",lambda x:np.mean(np.asarray(x)>0)),
        negative_fraction=("coef",lambda x:np.mean(np.asarray(x)<0))).reset_index())
S["sign_consistency"]=S[["positive_fraction","negative_fraction"]].max(axis=1)
S=S.sort_values(["target_name","nonzero_fraction","mean_abs_coef"],ascending=[True,False,False])
S.to_csv(OUT/"coefficient_stability_summary.csv",index=False)
for t in TARGET_NAMES:
    print("\\n",t); display(S[S.target_name==t].head(25))

,target_name,requested_fpr,metric,comparison,n_repeats,mean_delta,median_delta,q025,q975,fraction_gt_zero
0,FILD or FILA ADJUDICATED,0.005,sensitivity,genomics_plus_zebra - zebra,30,-0.061842,0.0,-0.181908,0.000000,0.000000
1,FILD or FILA ADJUDICATED,0.005,ppv,genomics_plus_zebra - zebra,30,-0.076630,0.0,-0.258160,0.000000,0.000000
2,FILD or FILA ADJUDICATED,0.010,sensitivity,genomics_plus_zebra - zebra,30,-0.053728,0.0,-0.165132,0.000000,0.000000
3,FILD or FILA ADJUDICATED,0.010,ppv,genomics_plus_zebra - zebra,30,-0.052496,0.0,-0.167023,0.000000,0.000000
4,FILD or FILA ADJUDICATED,0.050,sensitivity,genomics_plus_zebra - zebra,30,0.001096,0.0,-0.063651,0.046053,0.266667
5,FILD or FILA ADJUDICATED,0.050,ppv,genomics_plus_zebra - zebra,30,0.000181,0.0,-0.019376,0.013312,0.266667


\n FILD or FILA ADJUDICATED


,target_name,feature,n_splits,nonzero_fraction,mean_coef,median_coef,mean_abs_coef,max_abs_coef,positive_fraction,negative_fraction,sign_consistency
0,FILD or FILA ADJUDICATED,predicted_risk,30,1.000000,0.366415,0.368588,0.366415,0.447661,1.000000,0.0,1.000000
267,FILD or FILA ADJUDICATED,rs35705950.1_G_2,30,0.400000,-0.015134,0.000000,0.015134,0.077662,0.000000,0.4,0.400000
266,FILD or FILA ADJUDICATED,rs35705950.1_G_1,30,0.100000,0.002105,0.000000,0.002105,0.038783,0.100000,0.0,0.100000
67,FILD or FILA ADJUDICATED,rs117750427_A_0,30,0.066667,0.002232,0.000000,0.002232,0.033864,0.066667,0.0,0.066667
244,FILD or FILA ADJUDICATED,rs325093_T_0,30,0.033333,0.000854,0.000000,0.000854,0.025617,0.033333,0.0,0.033333
1,FILD or FILA ADJUDICATED,rs1015097_A_0,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
2,FILD or FILA ADJUDICATED,rs1015097_A_1,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
3,FILD or FILA ADJUDICATED,rs1015097_A_2,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
4,FILD or FILA ADJUDICATED,rs10422762_A_0,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
5,FILD or FILA ADJUDICATED,rs10422762_A_1,30,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000


\n NONFIBROTIC ILD ADJUDICATED


,target_name,feature,n_splits,nonzero_fraction,mean_coef,median_coef,mean_abs_coef,max_abs_coef,positive_fraction,negative_fraction,sign_consistency


\n NON FIBROTIC ILA ADJUDICATED


,target_name,feature,n_splits,nonzero_fraction,mean_coef,median_coef,mean_abs_coef,max_abs_coef,positive_fraction,negative_fraction,sign_consistency
